# 19 - Results & interpretability (Marsico/Gagneur house style)

**What.** The performance story (which head predicts the unseen-RBP nt-resolution eCLIP profile) and the **interpretability made concrete** (read the motif, see where on the RNA, decompose coarse vs fine), in the labs' figure idioms with domain-specific regulatory-genomics viz.

**Headline.** Per-residue cross-attention wins the profile; **`perres_aux` matches it (0.164 vs 0.159) while adding a faithful readable PWM** - so we get per-residue performance with an explicit motif readout. Every head that makes the PWM the *predictor* (BioPWM/Form-D/occ_footprint/motifscan) is null/below - interpretability comes from *explaining* the cross-attn, not bottlenecking it.

**Data (Moyon/Marsico lab).** Frozen all-223 PARNET `parnet.7m-0.0`, full-223 `encode.filtered.hfds` eCLIP, per-residue ESM. M2 = nt-resolution profile, leave-out-RBP zero-shot, **5-fold over RBPs** (every RBP held out once). Colorblind-safe ACGU logos; eCLIP tracks are softmax-over-positions (sum=1). Figures via `scripts/plot_style.py`.

## 1 - Performance: which head predicts the unseen-RBP profile?

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.set_house_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n; return json.loads(p.read_text()) if p.exists() else None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
b=J('m2_bakeoff_HepG2.json')
for k,v in sorted(b.items(),key=lambda kv:-kv[1]['gap_fam']): print(f"  {k:14} real {v['real']:.3f} | within-family gap {v['gap_fam']:+.4f} CI[{v['ci'][0]:+.4f},{v['ci'][1]:+.4f}]")

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.set_house_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n; return json.loads(p.read_text()) if p.exists() else None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
b=J('m2_bakeoff_HepG2.json')
kind={'per-residue':'winner','perres_aux':'match','BioPWM':'null','Form D':'null',"Form D'":'null','occ_footprint':'null','perrespwm':'below','motifscan':'below'}
items=[(k,v['gap_fam'],v['ci'][0],v['ci'][1],kind.get(k,'below')) for k,v in b.items()]
fig,(a,c)=plt.subplots(1,2,figsize=(10,3.8),gridspec_kw={'width_ratios':[1.2,1]})
ps.bakeoff_bars(a,items,null_level=0,noise=b['BioPWM']['ci'][1],ylabel='within-family profile-Pearson gap',title='Head bake-off (5-fold, HepG2)')
# leakage control battery from the per-residue k-fold rows
R=J('m2_kfold_HepG2.json')['archs']['perres']['rows']
ps.gap_violin(c,{'protein':[r['pearson_real'] for r in R],'shuffle':[r['pearson_shuf'] for r in R],'within-family':[r['pearson_fam'] for r in R]},ylabel='per-RBP profile Pearson',title='Leakage-controlled (per-residue)',paired=True)
ps.panel_label(a,'a'); ps.panel_label(c,'b')
show(fig,'nb19_perf.png','Per-residue wins; perres_aux matches; PWM-as-predictor heads are null/below')

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.set_house_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n; return json.loads(p.read_text()) if p.exists() else None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
h=J('m2_kfold_HepG2.json')['archs']['perres']['rows']; k=J('m2_kfold_K562.json')['archs']['perres']['rows']
hb={r['rbp']:r['pearson_real']-r['pearson_fam'] for r in h}; kb={r['rbp']:r['pearson_real']-r['pearson_fam'] for r in k}
common=[r for r in hb if r in kb]
fig,ax=plt.subplots(figsize=(3.6,3.6))
ps.scatter_identity(ax,[hb[r] for r in common],[kb[r] for r in common],xlabel='HepG2 within-family gap',ylabel='K562 within-family gap',title='Cross-cell replication')
show(fig,'nb19_replication.png')

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.set_house_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n; return json.loads(p.read_text()) if p.exists() else None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
b=J('m2_bakeoff_HepG2.json')
display(Markdown(f'''**Result (performance).** Per-residue cross-attn {b['per-residue']['real']:.3f} (within-family gap {b['per-residue']['gap_fam']:+.4f}); **perres_aux {b['perres_aux']['real']:.3f} / {b['perres_aux']['gap_fam']:+.4f} matches (CI overlaps)** while exposing a faithful PWM. Every PWM-as-predictor head is null on the profile (BioPWM {b['BioPWM']['gap_fam']:+.4f}, Form D {b['Form D']['gap_fam']:+.4f}, occ_footprint {b['occ_footprint']['gap_fam']:+.4f}) or below (perrespwm {b['perrespwm']['gap_fam']:+.4f}, motifscan {b['motifscan']['gap_fam']:+.4f}). The leakage-controlled gap (protein vs within-family shuffle) is the project's distinct, cross-cell-replicating selling point.'''))

## 2 - Read the motif (generated PWMs as information-content logos)

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.set_house_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n; return json.loads(p.read_text()) if p.exists() else None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
import numpy as np
z=None
for nm in ['biopwm_recog_indist_pwms.npz','biopwm_pwms_indist.npz']:
    p=OUT/nm
    if p.exists(): z=np.load(p,allow_pickle=True); break
if z is not None:
    P=z['pwm']; syms=list(z['syms'])
    if P.ndim==4: P=P.mean(1)
    ic=(2+(np.clip(P,1e-9,1)*np.log2(np.clip(P,1e-9,1))).sum(-1)).sum(-1)
    pick=list(np.argsort(-ic)[:6])
    fig=ps.logo_grid([P[i] for i in pick],[syms[i] for i in pick],ncol=3)
    show(fig,'nb19_logos.png','Generated motifs (read directly) - colorblind-safe bits logos')
else: display(Markdown('_PWMs unavailable_'))

Caveat (Gagneur discipline): motif recovery vs ATtRACT/RBPmap is only credible **with** a shuffle control (recovery-vs-mix_coeff collapsed +0.48->+0.08 on real weights). The PWM is the BioPWM's explicit latent; on the *profile* task it is interpretable but not the best predictor (Section 1).

## 3 - See where on the RNA (eCLIP coverage track, observed vs predicted)

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.set_house_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n; return json.loads(p.read_text()) if p.exists() else None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
import numpy as np
d=OUT/'m2_dump_zsdump_HepG2.npz'
if d.exists():
    z=np.load(d,allow_pickle=True); rbp=z['rbp']; syms=list(z['syms']); obs=z['obs'].astype(float); ptr=z['pt_real'].astype(float); pts=z['pt_shuf'].astype(float)
    pk=int(np.argmax(obs.sum(1)*(obs.max(1)>0)))   # a window with clear signal
    L=obs.shape[1]; pos=np.arange(L)-L//2
    fig,axes=plt.subplots(2,1,figsize=(6.2,3.2),sharex=True,gridspec_kw={'height_ratios':[1.0,1.4]})
    rn=syms[int(rbp[pk])] if int(rbp[pk])<len(syms) else str(rbp[pk])
    osh=obs[pk]/ (obs[pk].sum()+1e-9)
    pr=ps.np if False else None
    pe=float(((ptr[pk]-ptr[pk].mean())*(osh-osh.mean())).sum()/((ptr[pk].std()*osh.std()*L)+1e-9))
    ps.eclip_track(axes,pos,obs[pk],ptr[pk],shuffled=pts[pk],rbp=rn,pearson=pe)
    show(fig,'nb19_track.png','Predicted nt-resolution eCLIP profile tracks the observed coverage (unseen RBP)')
else: display(Markdown('_profile dump unavailable_'))

## 4 - Coarse vs fine: is the signal genuinely nt-resolution?

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.set_house_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n; return json.loads(p.read_text()) if p.exists() else None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
d=J('m2_decompose_zsdump_HepG2.json')
if d:
    rows=[{'rbp':r['rbp'][:9],'coarse_gap_shuf':r['coarse_gap_shuf'],'fine_gap_shuf':r['fine_gap_shuf']} for r in d['rows']]
    fig,ax=plt.subplots(figsize=(5.5,3.8)); ps.decomp_bars(ax,rows,title='Coarse envelope vs fine single-nt (per-RBP)')
    show(fig,'nb19_decomp.png','The protein gap has a real FINE single-nt component (nt-resolution)')
else: display(Markdown('_decomposition unavailable_'))

## 5 - Synthesis & honest limitations

| claim | status |
|---|---|
| Protein conditioning gives a real, leakage-controlled, cross-cell nt-resolution gap | **holds** (per-residue +0.025-0.031, p<1e-2, both cells) |
| An interpretable head can MATCH per-residue performance | **holds via `perres_aux`** (0.164, faithful PWM proxy) |
| Making the PWM the predictor matches per-residue | **false** (BioPWM/Form-D/occ_footprint null; perrespwm/motifscan below) |
| The win is genuinely nt-resolution (fine), not just coarse | **holds** (fine gap >0, Section 4) |
| Per-residue attention localizes to RRM/KH zero-shot | **false** (0.88x/0.55x below control - the honest limit; why the PWM proxy matters) |
| ProtT5 / 3Di / bigger protein helps | **false** (ProtT5 == ESM; protein-richness is a null lever) |

**Bottom line.** Per-residue cross-attention is the M2-profile mechanism; **`perres_aux` delivers its performance with an explicit, readable motif** (interpretation by faithful proxy, not by bottleneck). The leakage-controlled fine-resolution zero-shot gap is the CORAL-distinct contribution; the protein-structure and PWM-bottleneck routes are honestly closed. KB: cross-attention-architecture-investigation-2026-06-21, biopwm-unified-contract-2026-06-27, scaling-toward-coral-and-m2-zeroshot-2026-06-27.

Claude-assisted; figures in Marsico (RBPNet/PanRBPNet/RIBEX) + Gagneur (OUTRIDER/AbSplice) house style.